<a href="https://colab.research.google.com/github/Vedant2100/CS226-Final-Project/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

INSTALL DEPENDENCIES

In [10]:
!apt-get install -y gdal-bin libgdal-dev 2>/dev/null | tail -3
!pip install -q rasterio==1.3.10 shapely psycopg2-binary boto3 numpy tqdm
!pip install boto3 botocore

gdal-bin is already the newest version (3.8.4+dfsg-1~jammy0).
libgdal-dev is already the newest version (3.8.4+dfsg-1~jammy0).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [11]:
# from google.colab import drive
# drive.mount('/content/drive')

# import os
# DRIVE_SOURCE_DIR = '/content/drive/MyDrive/BigDataProject'

USE TO CONNECT TO AWS

In [23]:
from google.colab import userdata
AWS_ACCESS_KEY_ID     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')
AWS_REGION            = 'us-west-1'
S3_BUCKET_NAME        = userdata.get('S3_BUCKET_NAME')
print(f"AWS credentials loaded. Region: {AWS_REGION} | Bucket: {S3_BUCKET_NAME}")

DB_CONFIG = {
    'host':     userdata.get('DB_HOST'),
    'port':     5432,
    'database': 'postgres',
    'user':     'postgres',
    'password': userdata.get('DB_PASSWORD'),
    'connect_timeout': 10
}
S3_RAW_PREFIX = 'raw/'

LOCAL_WORK_DIR = '/tmp/vegetation_pipeline'
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

AWS credentials loaded. Region: us-west-1 | Bucket: vegetation-anomaly-cogs


In [43]:
import re
import datetime
import numpy as np
import rasterio
from rasterio.transform import array_bounds
from rasterio.warp import transform_bounds
from rasterio.crs import CRS
from shapely.geometry import box as shapely_box

def parse_hls_filename(filepath):
    filename = os.path.basename(filepath)
    name_no_ext = filename.replace('.tif', '').replace('.TIF', '')

    # Try to match HLS naming convention
    # Pattern: HLS.{S30|L30}.{tile}.{YYYY}{DOY}T{HHMMSS}.{version}[.{band}]
    hls_pattern = r'^HLS\.([SL]30)\.([A-Z0-9]+)\.(\d{4})(\d{3})T(\d{6})\.([\d\.v]+)(?:\.([A-Z0-9_]+))?$'
    match = re.match(hls_pattern, name_no_ext)

    if match:
        product   = match.group(1)
        tile      = match.group(2)
        year      = int(match.group(3))
        doy       = int(match.group(4))
        time_str  = match.group(5)
        version   = match.group(6)
        band      = match.group(7)

        acq_date = datetime.date(year, 1, 1) + datetime.timedelta(days=doy - 1)

        # Build tile_id that includes product and tile for uniqueness
        tile_id = f"{product}_{tile}_{year}{doy:03d}"

        return {
            'tile_id':          tile_id,
            'acquisition_date': acq_date,
            'product':          product,          # S30=Sentinel-2, L30=Landsat
            'tile':             tile,
            'band':             band or 'MULTI',
            'version':          version,
            'year':             year,
            'doy':              doy,
            'filename':         filename,
        }
    else:
        print(f"  [WARN] Non-HLS filename: {filename}. Attempting 'veg_YYYY-MM-DD' format.")
        veg_pattern = r'^veg_(\d{4})-(\d{2})-(\d{2})(?:\.cog)?$'
        veg_match = re.match(veg_pattern, name_no_ext)

        if veg_match:
            year = int(veg_match.group(1))
            month = int(veg_match.group(2))
            day = int(veg_match.group(3))
            acq_date = datetime.date(year, month, day)

            return {
                'tile_id':          name_no_ext,
                'acquisition_date': acq_date,
                'product':          'CUSTOM_VEG', # Indicate custom parsing
                'tile':             name_no_ext,
                'band':             'MULTI',
                'version':          'parsed',
                'year':             year,
                'doy':              acq_date.timetuple().tm_yday, # Calculate day of year
                'filename':         filename,
            }
        else:
            print(f"  [WARN] Unrecognized filename format: {filename}. Using generic fallback metadata.")
            # Original fallback logic for truly unrecognized formats
            year_match = re.search(r'(20[12][0-9])', filename)
            year = int(year_match.group(1)) if year_match else 2020
            return {
                'tile_id':          name_no_ext,
                'acquisition_date': datetime.date(int(datetime.datetime.now().year), 1, 1),
                'product':          'UNKNOWN',
                'tile':             name_no_ext,
                'band':             'MULTI',
                'version':          'unknown',
                'year':             year,
                'doy':              1,
                'filename':         filename,
            }

def extract_raster_metadata(filepath):
    metadata = {}

    with rasterio.open(filepath) as src:
        metadata['crs']        = str(src.crs)
        metadata['width_px']   = src.width
        metadata['height_px']  = src.height
        metadata['band_count'] = src.count
        metadata['resolution_m'] = abs(src.res[0])

        # Reproject bounds to WGS84 for PostGIS
        src_bounds = src.bounds
        if src.crs and str(src.crs) != 'EPSG:4326':
            try:
                wgs84_bounds = transform_bounds(
                    src.crs, CRS.from_epsg(4326),
                    src_bounds.left, src_bounds.bottom,
                    src_bounds.right, src_bounds.top
                )
                metadata['bbox_wgs84'] = {
                    'west':  wgs84_bounds[0],
                    'south': wgs84_bounds[1],
                    'east':  wgs84_bounds[2],
                    'north': wgs84_bounds[3],
                }
            except Exception as e:
                print(f"  [WARN] CRS reprojection failed: {e}. Using native bounds.")
                metadata['bbox_wgs84'] = {
                    'west':  src_bounds.left,  'south': src_bounds.bottom,
                    'east':  src_bounds.right, 'north': src_bounds.top,
                }
        else:
            metadata['bbox_wgs84'] = {
                'west':  src_bounds.left,  'south': src_bounds.bottom,
                'east':  src_bounds.right, 'north': src_bounds.top,
            }

        metadata['is_tiled']      = src.is_tiled
        metadata['compression']   = str(src.compression) if src.compression else 'None'
        metadata['has_overviews'] = len(src.overviews(1)) > 0

    return metadata

In [46]:
import subprocess
import time
import os
import rasterio
from rasterio.enums import Resampling

def validate_cog(filepath):
    result = {
        'is_valid': False,
        'is_tiled': False,
        'has_overviews': False,
        'compression': None,
        'issues': []
    }

    try:
        with rasterio.open(filepath) as src:
            result['is_tiled']     = src.is_tiled
            result['compression']  = str(src.compression) if src.compression else 'None'
            result['overviews']    = src.overviews(1)
            result['has_overviews'] = len(src.overviews(1)) > 0
            result['width']        = src.width
            result['height']       = src.height

            # Check for COG ghost metadata
            tags = src.tags()
            result['has_cog_marker'] = 'OVR_RESAMPLING_ALG' in tags or src.is_tiled

            if not result['is_tiled']:
                result['issues'].append("Not internally tiled (required for efficient S3 reads)")
            if not result['has_overviews']:
                result['issues'].append("No overviews (partial zoom reads will be slow)")
            if result['compression'] in ['None', None]:
                result['issues'].append("No compression (wastes S3 storage and transfer costs)")

        result['is_valid'] = len(result['issues']) == 0
    except Exception as e:
        result['issues'].append(f"Cannot open file: {e}")

    return result

def convert_to_cog(input_path, output_path, config):
    if not os.path.exists(input_path):
        return {'success': False, 'error': f"Input file not found: {input_path}"}

    compress  = config.get('cog_compress', 'LZW')
    blocksize = config.get('cog_blocksize', 512)

    cmd = [
        'gdal_translate', '-of', 'COG',
        '-co', f'COMPRESS={compress}',
        '-co', f'BLOCKSIZE={blocksize}',
        '-co', 'OVERVIEWS=AUTO',
        '-co', 'RESAMPLING=AVERAGE',
        '-co', 'BIGTIFF=IF_SAFER',
        '--config', 'GDAL_TIFF_OVR_BLOCKSIZE', str(blocksize),
        input_path, output_path
    ]

    print(f"  Converting to COG: {os.path.basename(input_path)}")
    start_time = time.time()

    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        duration = round(time.time() - start_time, 2)

        if proc.returncode != 0:
            print(f"  [ERROR] {proc.stderr[:200]}")
            return {'success': False, 'error': proc.stderr}

        input_mb  = os.path.getsize(input_path)  / 1e6
        output_mb = os.path.getsize(output_path) / 1e6
        print(f"  Done in {duration}s | {input_mb:.1f} MB -> {output_mb:.1f} MB "
              f"({input_mb/output_mb:.1f}x compression)")

        return {
            'success': True,
            'output_path': output_path,
            'duration_sec': duration,
            'input_size_mb': round(input_mb, 2),
            'file_size_mb': round(output_mb, 2),
        }

    except subprocess.TimeoutExpired:
        return {'success': False, 'error': 'Conversion timed out after 5 minutes'}
    except Exception as e:
        return {'success': False, 'error': str(e)}
def benchmark_cog_vs_original(original_path, cog_path, n_trials=3):
    results = {}

    for label, path in [('Original', original_path), ('COG', cog_path)]:
        if not os.path.exists(path):
            continue
        times = []
        with rasterio.open(path) as src:
            w, h = src.width, src.height
            # Read a 500x500 window from center
            window = rasterio.windows.Window(
                col_off=w // 4, row_off=h // 4,
                width=min(500, w // 2), height=min(500, h // 2)
            )
            for _ in range(n_trials):
                t0 = time.time()
                _ = src.read(1, window=window)
                times.append(time.time() - t0)

        results[label] = {
            'mean_ms': round(np.mean(times) * 1000, 1),
            'min_ms':  round(np.min(times) * 1000, 1),
        }

    if 'Original' in results and 'COG' in results:
        speedup = results['Original']['mean_ms'] / results['COG']['mean_ms']
        print(f"  Benchmark Results ({n_trials} trials):")
        print(f"    Original: {results['Original']['mean_ms']:.1f}ms avg")
        print(f"    COG:      {results['COG']['mean_ms']:.1f}ms avg")
        print(f"    Speedup:  {speedup:.1f}x faster with COG")
        print(f"  (On S3, this speedup is typically 10-50x for partial reads)")

    return results

In [45]:
import boto3
from botocore.exceptions import ClientError, NoCredentialsError


def get_s3_client(config):
    try:
        s3 = boto3.client(
            's3',
            aws_access_key_id     = config['aws_access_key_id'],
            aws_secret_access_key = config['aws_secret_access_key'],
            region_name           = config['aws_region']
        )
        # Quick connectivity test: list first page of bucket objects
        s3.list_objects_v2(Bucket=config['s3_bucket'], MaxKeys=1)
        print(f"  S3 connected: s3://{config['s3_bucket']} (region: {config['aws_region']})")
        return s3
    except NoCredentialsError:
        print("  [ERROR] Invalid AWS credentials. Check your Colab Secrets.")
        return None
    except ClientError as e:
        if e.response['Error']['Code'] == 'NoSuchBucket':
            print(f"  [ERROR] Bucket '{config['s3_bucket']}' does not exist.")
        else:
            print(f"  [ERROR] S3 connection failed: {e}")
        return None

def upload_cog_to_s3(local_path, config, s3_client, tile_metadata):
    year = tile_metadata.get('year', 2020)
    tile_id = tile_metadata.get('tile_id', 'unknown')
    filename = os.path.basename(local_path)

    # Build S3 key with year-based prefix
    s3_key = f"{config['s3_cog_prefix']}{year}/{tile_id}.cog.tif"
    s3_url = f"s3://{config['s3_bucket']}/{s3_key}"

    # Check if already uploaded (skip if exists)
    if config.get('skip_existing'):
        try:
            s3_client.head_object(Bucket=config['s3_bucket'], Key=s3_key)
            print(f"  [SKIP] Already in S3: {s3_url}")
            return s3_url
        except ClientError as e:
            if e.response['Error']['Code'] != '404':
                pass  # File doesn't exist, proceed with upload

    file_size_mb = os.path.getsize(local_path) / 1e6
    print(f"  Uploading {file_size_mb:.1f} MB -> {s3_url}")

    try:
        # Use upload_file for automatic multipart uploads (>8 MB)
        s3_client.upload_file(
            local_path,
            config['s3_bucket'],
            s3_key,
            ExtraArgs={
                'ContentType': 'image/tiff',
                'StorageClass': 'STANDARD',
                'Metadata': {
                    'tile-id':          tile_id,
                    'acquisition-date': str(tile_metadata.get('acquisition_date', '')),
                    'product':          tile_metadata.get('product', 'HLS'),
                }
            }
        )
        print(f"  Upload complete: {s3_url}")
        return s3_url

    except Exception as e:
        print(f"  [ERROR] S3 upload failed: {e}")
        return None

def download_from_s3(s3_key, local_dest, config, s3_client):
    local_path = os.path.join(local_dest, os.path.basename(s3_key))
    if os.path.exists(local_path):
        print(f"  [CACHE] Already downloaded: {os.path.basename(s3_key)}")
        return local_path

    try:
        obj = s3_client.head_object(Bucket=config['s3_bucket'], Key=s3_key)
        size_mb = obj['ContentLength'] / 1e6
        print(f"  Downloading {size_mb:.1f} MB from s3://{config['s3_bucket']}/{s3_key}")

        s3_client.download_file(config['s3_bucket'], s3_key, local_path)
        print(f"  Downloaded -> {local_path}")
        return local_path
    except Exception as e:
        print(f"  [ERROR] S3 download failed: {e}")
        return None

def list_s3_raw_files(config, s3_client):
    paginator = s3_client.get_paginator('list_objects_v2')
    pages = paginator.paginate(
        Bucket=config['s3_bucket'],
        Prefix=config['s3_raw_prefix']
    )
    keys = []
    for page in pages:
        for obj in page.get('Contents', []):
            key = obj['Key']
            if key.endswith('.tif') or key.endswith('.TIF'):
                keys.append(key)
    print(f"  Found {len(keys)} raw TIF files in s3://{config['s3_bucket']}/{config['s3_raw_prefix']}")
    return keys

In [44]:
import psycopg2
from psycopg2 import sql
import datetime

def get_db_connection(config):
    db_config = config['db_config']
    try:
        conn = psycopg2.connect(
            host=db_config['host'],
            port=db_config['port'],
            database=db_config['database'],
            user=db_config['user'],
            password=db_config['password'],
            connect_timeout=db_config.get('connect_timeout', 10)
        )
        conn.autocommit = False   # Use explicit transactions

        # Test PostGIS availability
        cur = conn.cursor()
        cur.execute("CREATE EXTENSION IF NOT EXISTS postgis;")
        cur.execute("SELECT PostGIS_Version();")
        postgis_ver = cur.fetchone()[0]
        cur.execute("SELECT version();")
        pg_ver = cur.fetchone()[0].split(' ')[1]

        print(f"  Database connected successfully!")
        print(f"  PostgreSQL version: {pg_ver}")
        print(f"  PostGIS version: {postgis_ver.split(' ')[0]}")
        cur.close()
        return conn

    except psycopg2.OperationalError as e:
        print(f"  [ERROR] Cannot connect to database: {e}")
        print(f"  Check: 1) RDS endpoint correct? 2) Security group allows port 5432?")
        print(f"         3) RDS instance is 'Available' in AWS console?")
        return None

def setup_database(conn):
    cur = conn.cursor()

    # Step 1: Enable PostGIS extensions
    print("  Setting up PostGIS extensions...")
    cur.execute("CREATE EXTENSION IF NOT EXISTS postgis;")
    cur.execute("CREATE EXTENSION IF NOT EXISTS postgis_raster;")
    conn.commit()

    # Step 2: Create main metadata table
    # This is the "Spatiotemporal Data Cube Index" from Section 4.4:
    #   - Space:  geom (GEOMETRY Polygon in WGS84)
    #   - Time:   acquisition_date (DATE)
    #   - Link:   file_url (S3 URL or local path to COG)
    print("  Creating vegetation_metadata table...")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS vegetation_metadata (
            id              SERIAL PRIMARY KEY,
            tile_id         TEXT NOT NULL,
            acquisition_date DATE NOT NULL,
            cloud_cover     FLOAT,
            ndvi_mean       FLOAT,
            ndvi_std        FLOAT,
            ndmi_mean       FLOAT,
            crs             TEXT,
            width_px        INTEGER,
            height_px       INTEGER,
            resolution_m    FLOAT,
            band_count      INTEGER,
            compression     TEXT,
            has_overviews   BOOLEAN,
            source          TEXT DEFAULT 'HLS',
            file_url        TEXT,           -- S3 URL or Drive path
            geom            GEOMETRY(Polygon, 4326),  -- WGS84 bounding box polygon
            inserted_at     TIMESTAMP DEFAULT NOW(),
            UNIQUE(tile_id)                -- Prevent duplicate scene insertions
        );
    """)
    conn.commit()

    # Step 3: Create GIST spatial index
    # This enables fast ST_Intersects queries (the core of the spatiotemporal filter)
    # Equivalent to the "CREATE INDEX ... USING GIST" in Section 4.3 of project outline
    print("  Creating GIST spatial index (idx_vegetation_geom)...")
    cur.execute("""
        CREATE INDEX IF NOT EXISTS idx_vegetation_geom
            ON vegetation_metadata
            USING GIST (geom);
    """)

    # Step 4: Create temporal B-tree index
    # Enables fast date range queries: WHERE acquisition_date BETWEEN '2020-01' AND '2020-12'
    print("  Creating temporal index (idx_vegetation_date)...")
    cur.execute("""
        CREATE INDEX IF NOT EXISTS idx_vegetation_date
            ON vegetation_metadata (acquisition_date);
    """)

    # Step 5: Create composite index for tile+date lookups
    cur.execute("""
        CREATE INDEX IF NOT EXISTS idx_vegetation_tile_date
            ON vegetation_metadata (tile_id, acquisition_date);
    """)

    conn.commit()
    print("  Database schema initialized!")

    # Verify setup
    cur.execute("""
        SELECT indexname, indexdef
        FROM pg_indexes
        WHERE tablename = 'vegetation_metadata'
        ORDER BY indexname;
    """)
    indexes = cur.fetchall()
    print(f"  Indexes created: {len(indexes)}")
    for name, defn in indexes:
        print(f"    - {name}")

    cur.close()
    return True

def insert_scene_metadata(conn, tile_meta, raster_meta, file_url, config):
    cur = conn.cursor()

    bbox = raster_meta.get('bbox_wgs84', {})
    west  = bbox.get('west',  0.0)
    south = bbox.get('south', 0.0)
    east  = bbox.get('east',  0.0)
    north = bbox.get('north', 0.0)

    try:
        cur.execute("""
            INSERT INTO vegetation_metadata (
                tile_id, acquisition_date, cloud_cover,
                ndvi_mean, ndvi_std, ndmi_mean,
                crs, width_px, height_px, resolution_m,
                band_count, compression, has_overviews,
                source, file_url,
                geom
            ) VALUES (
                %s, %s, %s,
                %s, %s, %s,
                %s, %s, %s, %s,
                %s, %s, %s,
                %s, %s,
                ST_MakeEnvelope(%s, %s, %s, %s, 4326)
            )
            ON CONFLICT (tile_id) DO UPDATE SET
                file_url        = EXCLUDED.file_url,
                ndvi_mean       = EXCLUDED.ndvi_mean,
                ndvi_std        = EXCLUDED.ndvi_std,
                has_overviews   = EXCLUDED.has_overviews,
                inserted_at     = NOW();
        """, (
            tile_meta.get('tile_id'),
            tile_meta.get('acquisition_date'),
            None,                                   # cloud_cover (not yet computed)
            raster_meta.get('ndvi_mean'),
            raster_meta.get('ndvi_std'),
            raster_meta.get('ndmi_mean'),
            raster_meta.get('crs'),
            raster_meta.get('width_px'),
            raster_meta.get('height_px'),
            raster_meta.get('resolution_m'),
            raster_meta.get('band_count'),
            raster_meta.get('compression'),
            raster_meta.get('has_overviews'),
            tile_meta.get('product', 'HLS'),
            file_url,
            west, south, east, north
        ))
        conn.commit()
        cur.close()
        return True

    except psycopg2.errors.UniqueViolation:
        conn.rollback()
        print(f"  [SKIP] Duplicate: {tile_meta.get('tile_id')} already in DB")
        cur.close()
        return False
    except Exception as e:
        conn.rollback()
        print(f"  [ERROR] Insert failed: {e}")
        cur.close()
        return False
def run_spatiotemporal_query(conn, west, south, east, north,
                              start_date, end_date):
    cur = conn.cursor()
    cur.execute("""
        SELECT
            tile_id,
            acquisition_date,
            file_url,
            ndvi_mean,
            resolution_m,
            ST_AsText(geom) AS bbox_wkt
        FROM vegetation_metadata
        WHERE
            ST_Intersects(
                geom,
                ST_MakeEnvelope(%s, %s, %s, %s, 4326)
            )
            AND acquisition_date BETWEEN %s AND %s
        ORDER BY acquisition_date;
    """, (west, south, east, north, start_date, end_date))

    results = cur.fetchall()
    cur.close()
    return results

In [ ]:
import glob
import traceback
import datetime

def run_pipeline(config):
    phase_label = 'Google Drive' if config['phase'] == 1 else 'AWS S3'
    print(f"{'='*65}\n  PIPELINE | Phase {config['phase']}: {phase_label} -> COG -> S3 + PostGIS\n{'='*65}")

    summary = {'total_found': 0, 'processed': 0, 'skipped': 0, 'failed': 0,
               'total_size_mb': 0, 'start_time': datetime.datetime.now()}

    # --- Connections ---
    s3_client = None
    if config.get('upload_to_s3') or config['phase'] == 2:
        s3_client = get_s3_client(config)
        if s3_client is None and config['phase'] == 2:
            print("[FATAL] Phase 2 requires S3. Aborting.")
            return summary

    conn = get_db_connection(config)
    if conn is None:
        print("[FATAL] Cannot connect to database. Aborting.")
        return summary

    setup_database(conn)

    # --- Discover files ---
    if config['phase'] == 1:
        source_dir = config['drive_source_dir']
        if not os.path.exists(source_dir):
            print(f"[ERROR] Drive source not found: {source_dir}")
            conn.close()
            return summary
        input_files = sorted(glob.glob(os.path.join(source_dir, '*.tif')) +
                             glob.glob(os.path.join(source_dir, '*.TIF')))
        for f in input_files:
            print(f"  {os.path.basename(f)} ({os.path.getsize(f)/1e6:.1f} MB)")
    else:
        input_files = list_s3_raw_files(config, s3_client)

    summary['total_found'] = len(input_files)
    print(f"  Total files to process: {len(input_files)}")

    if not input_files:
        conn.close()
        return summary

    # --- Process each file ---
    for idx, input_item in enumerate(input_files, 1):
        print(f"\n[{idx}/{len(input_files)}] {os.path.basename(str(input_item))}")
        local_raw_path = local_cog_path = None
        cleanup_raw = cleanup_cog = False

        try:
            # Get local file
            if config['phase'] == 1:
                local_raw_path = input_item
            else:
                local_raw_path = download_from_s3(input_item, config['local_work_dir'], config, s3_client)
                cleanup_raw = True

            if not local_raw_path or not os.path.exists(local_raw_path):
                summary['failed'] += 1
                continue

            # Parse filename
            tile_meta = parse_hls_filename(local_raw_path)
            tile_id   = tile_meta['tile_id']
            print(f"  {tile_id} | {tile_meta['acquisition_date']} | {tile_meta['product']}")

            # Skip if already indexed
            if config.get('skip_existing'):
                cur = conn.cursor()
                cur.execute("SELECT id FROM vegetation_metadata WHERE tile_id = %s", (tile_id,))
                if cur.fetchone():
                    print(f"  [SKIP] Already indexed")
                    summary['skipped'] += 1
                    cur.close()
                    continue
                cur.close()

            # Convert to COG if needed
            cog_info = validate_cog(local_raw_path)
            if cog_info['is_valid']:
                local_cog_path = local_raw_path
            else:
                print(f"  Issues: {'; '.join(cog_info['issues'])}")
                cog_filename   = os.path.basename(local_raw_path).replace('.tif', '.cog.tif')
                local_cog_path = os.path.join(config['local_work_dir'], cog_filename)
                cleanup_cog    = True
                conv_result    = convert_to_cog(local_raw_path, local_cog_path, config)
                if not conv_result['success']:
                    print(f"  [ERROR] {conv_result['error']}")
                    summary['failed'] += 1
                    continue
                benchmark_cog_vs_original(local_raw_path, local_cog_path, n_trials=2)

            # Extract metadata
            raster_meta = extract_raster_metadata(local_cog_path)
            bbox = raster_meta['bbox_wgs84']
            print(f"  BBox: W={bbox['west']:.4f} S={bbox['south']:.4f} "
                  f"E={bbox['east']:.4f} N={bbox['north']:.4f} | "
                  f"{raster_meta['width_px']}x{raster_meta['height_px']}px | "
                  f"{raster_meta['resolution_m']}m")

            # Upload to S3
            file_url = local_cog_path
            if config.get('upload_to_s3') and s3_client:
                file_url = upload_cog_to_s3(local_cog_path, config, s3_client, tile_meta) or local_cog_path

            # Index in PostGIS
            if insert_scene_metadata(conn, tile_meta, raster_meta, file_url,config):
                summary['processed']     += 1
                summary['total_size_mb'] += os.path.getsize(local_cog_path) / 1e6
                print(f"  [OK] {tile_id}")
            else:
                summary['failed'] += 1

        except Exception as e:
            print(f"  [ERROR] {e}")
            traceback.print_exc()
            summary['failed'] += 1
            if conn and not conn.closed:
                conn.rollback()

        finally:
            if cleanup_raw and local_raw_path and os.path.exists(local_raw_path):
                os.remove(local_raw_path)
            if cleanup_cog and local_cog_path and os.path.exists(local_cog_path):
                os.remove(local_cog_path)

    # --- Summary ---
    duration = (datetime.datetime.now() - summary['start_time']).total_seconds()
    print(f"\n{'='*65}\n  DONE in {duration:.1f}s | "
          f"Processed: {summary['processed']} | "
          f"Skipped: {summary['skipped']} | "
          f"Failed: {summary['failed']}\n{'='*65}")

    if conn and not conn.closed:
        cur = conn.cursor()
        cur.execute("SELECT COUNT(*), MIN(acquisition_date), MAX(acquisition_date) FROM vegetation_metadata")
        count, min_d, max_d = cur.fetchone()
        print(f"  DB: {count} scenes indexed | {min_d} to {max_d}")
        cur.close()
        conn.close()

    return summary

In [ ]:
CONFIG = {
    # AWS
    'aws_access_key_id':     AWS_ACCESS_KEY_ID,
    'aws_secret_access_key': AWS_SECRET_ACCESS_KEY,
    'aws_region':            "us-west-1",
    's3_bucket':             S3_BUCKET_NAME,
    's3_cog_prefix':         'cog/',          # Where COGs are stored in S3
    's3_raw_prefix':         S3_RAW_PREFIX,   # Where raw tifs are in S3 (Phase 2)

    # Database
    'db_config':             DB_CONFIG,

    # Processing
    'phase':                 1,
    'drive_source_dir':      DRIVE_SOURCE_DIR,
    'local_work_dir':        LOCAL_WORK_DIR,
    'upload_to_s3':          True,            # Upload COGs to S3 after conversion
    'skip_existing':         True,            # Skip if tile_id already in DB

    # COG conversion parameters
    'cog_compress':          'LZW',           # Options: LZW, DEFLATE, ZSTD
    'cog_blocksize':         512,             # Tile size in pixels (512x512)
    'cog_overviews':         [2, 4, 8, 16],  # Overview levels for multi-res access

    # NDVI computation (optional, computed during metadata extraction)
    'compute_ndvi_preview':  True,            # Compute NDVI stats during ingestion
    'ndvi_sample_size':      1000,            # Pixels to sample for NDVI stats (fast)
}

print("Starting Vegetation Anomaly Detection Pipeline...")
print("This will:")
print("  1. Connect to AWS S3 and RDS PostgreSQL")
print("  2. Find all .tif files in your Google Drive source folder")
print("  3. Convert each file to COG format")
print("  4. Upload COGs to S3")
print("  5. Index metadata in PostGIS with spatial + temporal indexes")
print("")

# Run the complete pipeline
summary = run_pipeline(CONFIG)

print(f"\\nPipeline returned: {summary['processed']} scenes successfully indexed")

In [ ]:
print("=" * 65)
print("  SPATIAL QUERY VERIFICATION")
print("=" * 65)

# Reconnect for verification
conn_verify = get_db_connection(config=CONFIG)
if conn_verify:

    # --- Query 1: Show all indexed scenes ---
    cur = conn_verify.cursor()
    cur.execute("""
        SELECT
            tile_id,
            acquisition_date,
            ROUND(resolution_m::numeric, 1) AS res_m,
            ROUND(ndvi_mean::numeric, 3) AS ndvi,
            compression,
            has_overviews,
            LEFT(file_url, 60) AS file_url_preview,
            ST_AsText(ST_Envelope(geom)) AS bbox_wkt
        FROM vegetation_metadata
        ORDER BY acquisition_date;
    """)
    rows = cur.fetchall()
    cur.close()

    print(f"\\n[QUERY 1] All indexed scenes ({len(rows)} total):")
    print(f"  {'tile_id':<30} {'date':<12} {'res':<6} {'ndvi':<7} {'compress':<10} {'ovr':<5}")
    print(f"  {'-'*30} {'-'*12} {'-'*6} {'-'*7} {'-'*10} {'-'*5}")
    for r in rows:
        ndvi_str = f"{r[3]:.3f}" if r[3] is not None else "N/A  "
        print(f"  {str(r[0]):<30} {str(r[1]):<12} {str(r[2]):<6} {ndvi_str:<7} {str(r[4]):<10} {str(r[5]):<5}")

    # --- Query 2: Spatiotemporal range query (Boulder County) ---
    # Boulder County, Colorado approximate bounds:
    # West: -105.7, South: 39.9, East: -105.0, North: 40.3
    boulder_bounds = (-105.7, 39.9, -105.0, 40.3)
    start_date = datetime.date(2015, 1, 1)
    end_date   = datetime.date(2025, 12, 31)

    print(f"\\n[QUERY 2] Spatiotemporal query - Boulder County, CO (2015-2025):")
    print(f"  Bounds: W={boulder_bounds[0]}, S={boulder_bounds[1]}, "
          f"E={boulder_bounds[2]}, N={boulder_bounds[3]}")

    results = run_spatiotemporal_query(
        conn_verify,
        *boulder_bounds,
        start_date, end_date
    )

    print(f"  Scenes overlapping Boulder County in range: {len(results)}")
    if results:
        print(f"  These scene URLs will be passed to PySpark for NDVI computation:")
        for r in results[:5]:
            url = str(r[2])[:70] + ('...' if len(str(r[2])) > 70 else '')
            print(f"    {r[0]} | {r[1]} | NDVI={r[3]} | {url}")

    # --- Query 3: EXPLAIN ANALYZE - verify GIST index is used ---
    print(f"\\n[QUERY 3] EXPLAIN ANALYZE - Verify GIST index usage:")
    cur = conn_verify.cursor()
    cur.execute("""
        EXPLAIN (ANALYZE, FORMAT TEXT)
        SELECT tile_id, file_url FROM vegetation_metadata
        WHERE ST_Intersects(geom, ST_MakeEnvelope(-105.7, 39.9, -105.0, 40.3, 4326));
    """)
    plan = cur.fetchall()
    cur.close()
    print("  Query Plan:")
    for line in plan[:10]:
        print(f"    {line[0]}")
    print("  (Look for 'Index Scan using idx_vegetation_geom' = GIST index is being used!)")

    # --- Query 4: Count scenes by year ---
    cur = conn_verify.cursor()
    cur.execute("""
        SELECT
            EXTRACT(YEAR FROM acquisition_date)::int AS year,
            COUNT(*) AS scene_count,
            ROUND(AVG(ndvi_mean)::numeric, 3) AS avg_ndvi,
            MIN(acquisition_date) AS first_date,
            MAX(acquisition_date) AS last_date
        FROM vegetation_metadata
        GROUP BY year
        ORDER BY year;
    """)
    yearly = cur.fetchall()
    cur.close()

    if yearly:
        print(f"\\n[QUERY 4] Scenes indexed by year:")
        print(f"  {'Year':<6} {'Scenes':<8} {'Avg NDVI':<10} {'First':<12} {'Last':<12}")
        print(f"  {'-'*6} {'-'*8} {'-'*10} {'-'*12} {'-'*12}")
        for row in yearly:
            ndvi = f"{row[2]:.3f}" if row[2] else "N/A"
            print(f"  {row[0]:<6} {row[1]:<8} {ndvi:<10} {str(row[3]):<12} {str(row[4]):<12}")

    conn_verify.close()
    print("\\nVerification complete. The spatiotemporal index is operational!")
    print("Next step: Use these scene URLs in PySpark for NDVI/NDMI computation (Section 5).")

#### Data Transformation

In [13]:
!pip install -q pyspark apache-sedona

In [14]:
from google.colab import userdata
AWS_ACCESS_KEY_ID     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')
print('AWS credentials loaded.')

AWS credentials loaded.


In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, month, year, avg

spark = SparkSession.builder \
    .appName("VegetationPipeline") \
    .config("spark.driver.memory", "4g") \
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID) \
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com") \
    .getOrCreate()
print(f"Spark initialized: {spark.version}")

Spark initialized: 4.0.2


In [16]:
# Tile N39W108 covers lat 39-42°N, lon -108 to -105°W (includes Boulder County)
!curl -L \
https://esa-worldcover.s3.eu-central-1.amazonaws.com/v100/2020/map/ESA_WorldCover_10m_2020_v100_N39W108_Map.tif \
-o /tmp/worldcover_boulder.tif
print('WorldCover tile downloaded to /tmp/worldcover_boulder.tif')

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 85.2M  100 85.2M    0     0  16.6M      0  0:00:05  0:00:05 --:--:-- 18.7M
WorldCover tile downloaded to /tmp/worldcover_boulder.tif


In [17]:
import numpy as np
import boto3
import rasterio
from rasterio.session import AWSSession
from rasterio.transform import Affine

def read_bands_from_s3(results, aws_key, aws_secret, region='us-west-1',
                        red_band=3, nir_band=4, swir1_band=5, pixel_stride=10):
    """
    Reads Red, NIR, and SWIR1 band arrays directly from S3 COGs for each
    scene in results. Streams via rasterio AWSSession — no local download.

    Returns list of (tile_id, acq_date, red, nir, swir1, bbox_wkt,
                      strided_transform, crs_str) per scene.
    strided_transform and crs_str are needed by apply_forest_mask.
    """
    boto3_session = boto3.Session(
        aws_access_key_id=aws_key,
        aws_secret_access_key=aws_secret,
        region_name=region,
    )
    aws_session = AWSSession(boto3_session)
    scene_bands = []
    for tile_id, acq_date, file_url, _, _res, bbox_wkt in results:
        print(f'  Reading {tile_id}  ->  {file_url}')
        with rasterio.Env(aws_session):
            with rasterio.open(file_url) as src:
                n = src.count
                print(f'    bands={n}  shape={src.height}x{src.width}  crs={src.crs}')
                rb = min(red_band,   n)
                nb = min(nir_band,   n)
                sb = min(swir1_band, n)
                red   = src.read(rb)[::pixel_stride, ::pixel_stride].astype('float32')
                nir   = src.read(nb)[::pixel_stride, ::pixel_stride].astype('float32')
                swir1 = src.read(sb)[::pixel_stride, ::pixel_stride].astype('float32')
                nodata = src.nodata if src.nodata is not None else 0
                for arr in (red, nir, swir1):
                    arr[arr == nodata] = np.nan
                # Adjust affine transform for the stride so apply_forest_mask
                # can reproject WorldCover onto this exact subsampled grid
                t = src.transform
                strided_transform = Affine(
                    t.a * pixel_stride, t.b, t.c,
                    t.d, t.e * pixel_stride, t.f,
                )
                crs_str = src.crs
        print(f'    red   [{np.nanmin(red):.0f}, {np.nanmax(red):.0f}]')
        print(f'    nir   [{np.nanmin(nir):.0f}, {np.nanmax(nir):.0f}]')
        print(f'    swir1 [{np.nanmin(swir1):.0f}, {np.nanmax(swir1):.0f}]')
        scene_bands.append((tile_id, acq_date, red, nir, swir1, bbox_wkt,
                             strided_transform, crs_str))
    return scene_bands


In [47]:
import datetime
import psycopg2
from psycopg2 import sql

# Redefine necessary variables for CONFIG, including userdata calls
from google.colab import userdata
import os

AWS_ACCESS_KEY_ID     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')
AWS_REGION            = 'us-west-1'
S3_BUCKET_NAME        = userdata.get('S3_BUCKET_NAME')

DB_CONFIG = {
    'host':     userdata.get('DB_HOST'),
    'port':     5432,
    'database': 'postgres',
    'user':     'postgres',
    'password': userdata.get('DB_PASSWORD'),
    'connect_timeout': 10
}
S3_RAW_PREFIX = 'raw/'

LOCAL_WORK_DIR = '/tmp/vegetation_pipeline'
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

# Placeholder for DRIVE_SOURCE_DIR, as it's not strictly needed for this cell's function
# and is commented out in its original cell.
DRIVE_SOURCE_DIR = '/content/drive/MyDrive/BigDataProject' # Assuming this default path if needed elsewhere

# Redefine CONFIG dictionary
CONFIG = {
    'aws_access_key_id':     AWS_ACCESS_KEY_ID,
    'aws_secret_access_key': AWS_SECRET_ACCESS_KEY,
    'aws_region':            AWS_REGION,
    's3_bucket':             S3_BUCKET_NAME,
    's3_cog_prefix':         'cog/',
    's3_raw_prefix':         S3_RAW_PREFIX,
    'db_config':             DB_CONFIG,
    'phase':                 1, # Defaulting to phase 1, though not directly used here.
    'drive_source_dir':      DRIVE_SOURCE_DIR,
    'local_work_dir':        LOCAL_WORK_DIR,
    'upload_to_s3':          True,
    'skip_existing':         True,
    'cog_compress':          'LZW',
    'cog_blocksize':         512,
    'cog_overviews':         [2, 4, 8, 16],
    'compute_ndvi_preview':  True,
    'ndvi_sample_size':      1000,
}

def get_db_connection(config):
    db_config = config['db_config']
    try:
        conn = psycopg2.connect(
            host=db_config['host'],
            port=db_config['port'],
            database=db_config['database'],
            user=db_config['user'],
            password=db_config['password'],
            connect_timeout=db_config.get('connect_timeout', 10)
        )
        conn.autocommit = False   # Use explicit transactions

        # Test PostGIS availability
        cur = conn.cursor()
        cur.execute("CREATE EXTENSION IF NOT EXISTS postgis;")
        cur.execute("SELECT PostGIS_Version();")
        postgis_ver = cur.fetchone()[0]
        cur.execute("SELECT version();")
        pg_ver = cur.fetchone()[0].split(' ')[1]

        print(f"  Database connected successfully!")
        print(f"  PostgreSQL version: {pg_ver}")
        print(f"  PostGIS version: {postgis_ver.split(' ')[0]}")
        cur.close()
        return conn

    except psycopg2.OperationalError as e:
        print(f"  [ERROR] Cannot connect to database: {e}")
        print(f"  Check: 1) RDS endpoint correct? 2) Security group allows port 5432?")
        print(f"         3) RDS instance is 'Available' in AWS console?")
        return None

def run_spatiotemporal_query(conn, west, south, east, north,
                              start_date, end_date):
    cur = conn.cursor()
    cur.execute("""
        SELECT
            tile_id,
            acquisition_date,
            file_url,
            ndvi_mean,
            resolution_m,
            ST_AsText(geom) AS bbox_wkt
        FROM vegetation_metadata
        WHERE
            ST_Intersects(
                geom,
                ST_MakeEnvelope(%s, %s, %s, %s, 4326)
            )
            AND acquisition_date BETWEEN %s AND %s
        ORDER BY acquisition_date;
    """, (west, south, east, north, start_date, end_date))

    results = cur.fetchall()
    cur.close()
    return results

# Establish DB connection using the CONFIG dictionary defined earlier
conn_transform = get_db_connection(config=CONFIG)
if conn_transform:
    # Query for all COG URLs. Use a very wide spatial and temporal range
    # to capture all indexed scenes.
    all_bounds = (-180.0, -90.0, 180.0, 90.0) # Global extent
    all_start_date = datetime.date(1900, 1, 1) # Start of modern remote sensing
    all_end_date   = datetime.date.today() + datetime.timedelta(days=365) # Extend to future proof

    print("Fetching all COG URLs from PostGIS database...")
    results = run_spatiotemporal_query(
        conn_transform,
        *all_bounds,
        all_start_date, all_end_date
    )
    conn_transform.close()
    print(f'Found {len(results)} scene(s) in the database.')
else:
    print('Failed to connect to database. Cannot fetch scene URLs.')
    results = [] # Ensure results is an empty list if DB connection fails

if results:
    scene_bands = read_bands_from_s3(
        results,
        AWS_ACCESS_KEY_ID,
        AWS_SECRET_ACCESS_KEY,
        region=AWS_REGION,
        red_band=1, nir_band=2, swir1_band=3,  # Adjusted band assignments
        pixel_stride=10,   # set to 1 for full resolution
    )
    print(f'Bands loaded from S3 for {len(scene_bands)} scene(s). No download needed.')
else:
    print('No results found or database connection failed. Skipping band reading.')
    scene_bands = [] # Ensure scene_bands is defined.

  Database connected successfully!
  PostgreSQL version: 17.6
  PostGIS version: 3.5
Fetching all COG URLs from PostGIS database...
Found 67 scene(s) in the database.
  Reading veg_2023-10-01  ->  s3://vegetation-anomaly-cogs/cog/2023/veg_2023-10-01.cog.tif
    bands=3  shape=1073x1714  crs=EPSG:4326
    red   [1, 8296]
    nir   [2, 8069]
    swir1 [2, 11434]
  Reading veg_2023-10-08  ->  s3://vegetation-anomaly-cogs/cog/2023/veg_2023-10-08.cog.tif
    bands=3  shape=1073x1714  crs=EPSG:4326
    red   [2, 8034]
    nir   [0, 8066]
    swir1 [6, 7091]
  Reading veg_2023-10-15  ->  s3://vegetation-anomaly-cogs/cog/2023/veg_2023-10-15.cog.tif
    bands=3  shape=1073x1714  crs=EPSG:4326
    red   [4, 8062]
    nir   [0, 8030]
    swir1 [2, 6244]
  Reading veg_2023-10-22  ->  s3://vegetation-anomaly-cogs/cog/2023/veg_2023-10-22.cog.tif
    bands=3  shape=1073x1714  crs=EPSG:4326
    red   [1, 16179]
    nir   [1, 15672]
    swir1 [5, 15158]
  Reading veg_2023-10-29  ->  s3://vegetation-ano

In [48]:
def build_pixel_dataframe(sedona, scene_bands):
    """
    Flattens per-scene band arrays into a Spark DataFrame with one row per pixel.
    Columns: tile_id, date, red, nir, swir1, pixel_id, bbox_wkt
    NaN pixels (incl. non-forest pixels zeroed by apply_forest_mask) are dropped.
    Accepts both 6- and 8-tuple scene entries — extra fields (transform, crs) ignored.
    """
    pixel_rows = []
    for scene in scene_bands:
        tile_id, acq_date, red, nir, swir1, bbox_wkt = scene[:6]
        h, w = red.shape
        for r in range(h):
            for c in range(w):
                rv, nv, sv = float(red[r, c]), float(nir[r, c]), float(swir1[r, c])
                if rv != rv or nv != nv or sv != sv:  # NaN check
                    continue
                pixel_rows.append((
                    tile_id, acq_date,
                    rv, nv, sv,
                    f'{tile_id}_{r}_{c}',
                    bbox_wkt,
                ))
    print(f'  Pixel rows (non-NaN): {len(pixel_rows):,}')
    return sedona.createDataFrame(
        pixel_rows,
        ['tile_id', 'date', 'red', 'nir', 'swir1', 'pixel_id', 'bbox_wkt'],
    )


In [49]:
from rasterio.warp import reproject, Resampling

def apply_forest_mask(scene_bands, worldcover_path, forest_class=10):
    """
    Masks each scene's band arrays to forested pixels only using ESA WorldCover.
    Class 10 = Tree cover. Non-forest pixels are set to NaN.

    Operates on numpy arrays (before Spark) by reprojecting the WorldCover
    GeoTIFF onto each scene's subsampled pixel grid using rasterio.
    No Sedona, no geoparquet, no geom column required.

    Parameters
    ----------
    scene_bands     : list of 8-tuples from read_bands_from_s3()
                      (..., strided_transform, crs_str)
    worldcover_path : local path to the ESA WorldCover GeoTIFF
    forest_class    : WorldCover class for tree cover (default 10)
    """
    masked = []
    with rasterio.open(worldcover_path) as wc:
        for scene in scene_bands:
            tile_id, acq_date, red, nir, swir1, bbox_wkt, transform, crs = scene
            h, w = red.shape
            forest_mask = np.zeros((h, w), dtype=np.uint8)
            reproject(
                source=rasterio.band(wc, 1),
                destination=forest_mask,
                dst_transform=transform,
                dst_crs=crs,
                resampling=Resampling.nearest,
            )
            is_forest = (forest_mask == forest_class)
            for arr in (red, nir, swir1):
                arr[~is_forest] = np.nan
            n_forest = int(is_forest.sum())
            print(f'  [{tile_id}] Forest pixels: {n_forest:,} / {h*w:,} '
                  f'({100*n_forest/max(h*w,1):.1f}%)')
            masked.append((tile_id, acq_date, red, nir, swir1, bbox_wkt, transform, crs))
    return masked


In [50]:
def compute_vegetation_indices(df):
    """
    Computes NDVI and NDMI from Red, NIR, and SWIR1 bands.
    NDVI = (NIR - Red)  / (NIR + Red)
    NDMI = (NIR - SWIR1) / (NIR + SWIR1)
    """
    return df.withColumn('ndvi', (col('nir') - col('red'))   / (col('nir') + col('red'))) \
             .withColumn('ndmi', (col('nir') - col('swir1')) / (col('nir') + col('swir1')))


In [51]:
def harmonize_time_series(df):
    """
    Aggregates irregular observations into monthly composites.
    Returns a regular time series for each pixel.
    """
    return (
        df.groupBy('pixel_id', year('date').alias('year'), month('date').alias('month'))
          .agg(avg('ndvi').alias('ndvi_mean'), avg('ndmi').alias('ndmi_mean'))
          .orderBy('year', 'month')
    )


In [52]:
if 'scene_bands' in locals() and scene_bands:
    forest_data_path = '/tmp/worldcover_boulder.tif'
    masked_bands = apply_forest_mask(scene_bands, forest_data_path)
    spark_df   = build_pixel_dataframe(spark, masked_bands)
    indices_df = compute_vegetation_indices(spark_df)
    final_df   = harmonize_time_series(indices_df)
    final_df.show(10)
    print(f'Transformation Complete: {final_df.count()} monthly records generated.')
else:
    print('No scene_bands found. Run the S3 band-read cell above first.')

  [veg_2023-10-01] Forest pixels: 7,534 / 18,576 (40.6%)
  [veg_2023-10-08] Forest pixels: 7,534 / 18,576 (40.6%)
  [veg_2023-10-15] Forest pixels: 7,534 / 18,576 (40.6%)
  [veg_2023-10-22] Forest pixels: 7,534 / 18,576 (40.6%)
  [veg_2023-10-29] Forest pixels: 7,534 / 18,576 (40.6%)
  [veg_2023-11-05] Forest pixels: 7,534 / 18,576 (40.6%)
  [veg_2023-11-12] Forest pixels: 7,534 / 18,576 (40.6%)
  [veg_2023-11-19] Forest pixels: 7,534 / 18,576 (40.6%)
  [veg_2023-11-26] Forest pixels: 7,534 / 18,576 (40.6%)
  [veg_2023-12-03] Forest pixels: 7,534 / 18,576 (40.6%)
  [veg_2023-12-17] Forest pixels: 7,534 / 18,576 (40.6%)
  [veg_2023-12-24] Forest pixels: 7,534 / 18,576 (40.6%)
  [veg_2023-12-31] Forest pixels: 7,534 / 18,576 (40.6%)
  [veg_2024-01-07] Forest pixels: 7,534 / 18,576 (40.6%)
  [veg_2024-01-21] Forest pixels: 7,534 / 18,576 (40.6%)
  [veg_2024-01-28] Forest pixels: 7,534 / 18,576 (40.6%)
  [veg_2024-02-04] Forest pixels: 7,534 / 18,576 (40.6%)
  [veg_2024-02-11] Forest pixel

In [39]:
final_df.head(5)

[Row(pixel_id='veg_2023-10-08_2_34', year=2026, month=1, ndvi_mean=0.4943639291465378, ndmi_mean=-0.010133333333333333),
 Row(pixel_id='veg_2023-10-01_53_111', year=2026, month=1, ndvi_mean=0.6968503937007874, ndmi_mean=0.2762807225347942),
 Row(pixel_id='veg_2023-10-08_7_64', year=2026, month=1, ndvi_mean=0.5579937304075235, ndmi_mean=-0.005005005005005005),
 Row(pixel_id='veg_2023-10-01_63_106', year=2026, month=1, ndvi_mean=0.6809024979854955, ndmi_mean=0.17169069462647443),
 Row(pixel_id='veg_2023-10-08_13_43', year=2026, month=1, ndvi_mean=0.5157515751575158, ndmi_mean=-0.016929363689433742)]

In [41]:

# Z-Score Anomaly Detection
from pyspark.sql.functions import stddev, mean as spark_mean, when, col

# Historical baseline (mean & std per pixel per month)
baseline_df = final_df.groupBy('pixel_id', 'month').agg(
    spark_mean('ndvi_mean').alias('ndvi_mu'),
    stddev('ndvi_mean').alias('ndvi_sigma'),
    spark_mean('ndmi_mean').alias('ndmi_mu'),
    stddev('ndmi_mean').alias('ndmi_sigma'),
)

# Join with baseline and compute Z-scores
anomaly_df = final_df.join(baseline_df, on=['pixel_id', 'month'])

anomaly_df = anomaly_df.withColumn(
    'ndvi_zscore',
    when(col('ndvi_sigma') > 0,
         (col('ndvi_mean') - col('ndvi_mu')) / col('ndvi_sigma'))
    .otherwise(0.0)
).withColumn(
    'ndmi_zscore',
    when(col('ndmi_sigma') > 0,
         (col('ndmi_mean') - col('ndmi_mu')) / col('ndmi_sigma'))
    .otherwise(0.0)
)

# Flag anomalies (Z < -2.0)
THRESHOLD = -1.5
anomaly_df = anomaly_df.withColumn(
    'is_ndvi_anomaly', col('ndvi_zscore') < THRESHOLD
).withColumn(
    'is_ndmi_anomaly', col('ndmi_zscore') < THRESHOLD
)

# Show flagged anomalies
anomalies = anomaly_df.filter(col('is_ndvi_anomaly') | col('is_ndmi_anomaly'))
anomalies.show(20)
print(f'Total anomalies detected: {anomalies.count()}')
print(f'Total records evaluated: {anomaly_df.count()}')

+--------+-----+----+---------+---------+-------+----------+-------+----------+-----------+-----------+---------------+---------------+
|pixel_id|month|year|ndvi_mean|ndmi_mean|ndvi_mu|ndvi_sigma|ndmi_mu|ndmi_sigma|ndvi_zscore|ndmi_zscore|is_ndvi_anomaly|is_ndmi_anomaly|
+--------+-----+----+---------+---------+-------+----------+-------+----------+-----------+-----------+---------------+---------------+
+--------+-----+----+---------+---------+-------+----------+-------+----------+-----------+-----------+---------------+---------------+

Total anomalies detected: 0
Total records evaluated: 263693
